# From Combine Burst to Rookie Movement

**GridironIQ | NFL Big Data Bowl 2027**

This notebook tests whether player-only Combine sensor peak speed during the defensive-line pass-rush drill is associated with mean speed observed in that player's rookie regular season. It is an exploratory linkage analysis, not a prediction submission or causal study.

The cohort is selected by complete-case coverage only. Combine frames are reduced to attempt-level features; game tracking is processed in 200,000-row chunks and restricted to the prospect's draft-year regular-season games. No large game-tracking CSV is loaded into memory at once.

Attach the official NFL Big Data Bowl 2027 dataset as the notebook input. The code discovers the directory from Kaggle's `/kaggle/input` mount or from the current working directory. It uses only pandas, NumPy, and Matplotlib, which are available in Kaggle's standard Python image.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

OFFICIAL_SIZES = {
    'combine_results.csv': 41020,
    'combine_tracking.csv': 83788174,
    'game_tracking_2023.csv': 335463637,
    'game_tracking_2024.csv': 717952893,
    'game_tracking_2025.csv': 1022575093,
    'games.csv': 43174,
    'player_career_successes.csv': 13801,
    'player_play.csv': 146879042,
    'players.csv': 47301,
}

def find_data_directory():
    roots = []
    configured = os.environ.get('GRIDIRON_IQ_DATA_DIR')
    if configured:
        roots.append(Path(configured))
    roots.extend([Path('/kaggle/input'), Path.cwd()])
    visited = set()
    for root in roots:
        if not root.exists():
            continue
        root = root.resolve()
        if root in visited:
            continue
        visited.add(root)
        direct = root / 'combine_results.csv'
        if all((root / name).is_file() for name in OFFICIAL_SIZES):
            return root
        for match in root.rglob('combine_results.csv'):
            if all((match.parent / name).is_file() for name in OFFICIAL_SIZES):
                return match.parent
    raise FileNotFoundError(
        'Official competition CSVs were not found. Attach the Big Data Bowl 2027 '
        'dataset as notebook input or set GRIDIRON_IQ_DATA_DIR.'
    )

DATA_DIR = find_data_directory()
missing_files = [name for name in OFFICIAL_SIZES if not (DATA_DIR / name).is_file()]
assert not missing_files, f'Missing official data files: {missing_files}'
inventory = pd.DataFrame([
    {
        'file': name,
        'bytes': (DATA_DIR / name).stat().st_size,
        'official_bytes': size,
        'size_matches': (DATA_DIR / name).stat().st_size == size,
        'columns': len(pd.read_csv(DATA_DIR / name, nrows=0).columns),
    }
    for name, size in OFFICIAL_SIZES.items()
])
assert inventory['size_matches'].all(), 'One or more file sizes differ from the official inventory.'

combine = pd.read_csv(
    DATA_DIR / 'combine_results.csv',
    usecols=['draft_year', 'nfl_id', 'combine_position', 'forty'],
)
assert combine[['draft_year', 'nfl_id']].notna().all().all()
assert not combine.duplicated(['draft_year', 'nfl_id']).any()
games = pd.read_csv(DATA_DIR / 'games.csv', usecols=['game_id', 'season', 'season_type'])
assert games['game_id'].notna().all() and not games['game_id'].duplicated().any()
career = pd.read_csv(
    DATA_DIR / 'player_career_successes.csv',
    usecols=['nfl_id', 'career_offensive_snaps', 'career_defensive_snaps', 'career_special_teams_snaps'],
)
assert not career['nfl_id'].duplicated().any()

print(f'Data directory: {DATA_DIR}')
display(inventory)
print(f'Combine prospects: {len(combine):,}; unique IDs: {combine.nfl_id.nunique():,}')

## Build Combine and rookie-season features

Only `PLAYER` Combine rows are used; `BALL` rows share event times but are not athletes. Each attempt is ordered by its timestamp, summarized by its maximum observed speed, and player/drill peak speed is the median across attempts. Rookie outcomes use `games.csv` to identify `REG` games in the player's draft-year season. Speeds and accelerations are frame-weighted averages, so they reflect exposure and role as well as movement ability.

In [ ]:
COMBINE_COLUMNS = [
    'draft_year', 'nfl_id', 'entity_type', 'time', 'drill_type',
    'drill_name', 'attempt', 'x', 'y', 's', 'a',
]
sensor = pd.read_csv(DATA_DIR / 'combine_tracking.csv', usecols=COMBINE_COLUMNS)
sensor = sensor.loc[sensor['entity_type'].eq('PLAYER')].copy()
assert not sensor.empty, 'No PLAYER rows were found in Combine tracking.'
keys = ['draft_year', 'nfl_id', 'drill_type', 'drill_name', 'attempt']
assert sensor[keys].notna().all().all(), 'Combine player/drill/attempt keys are missing.'
sensor['time'] = pd.to_datetime(sensor['time'], errors='coerce')
assert sensor['time'].notna().all(), 'Combine tracking has invalid timestamps.'
for column in ['x', 'y', 's', 'a']:
    sensor[column] = pd.to_numeric(sensor[column], errors='coerce')
sensor = sensor.dropna(subset=['x', 'y']).sort_values([*keys, 'time'], kind='mergesort')
attempts = sensor.groupby(keys, sort=False, observed=True).agg(
    attempt_peak_speed=('s', 'max'),
    attempt_mean_acceleration=('a', 'mean'),
).reset_index()
combine_features = attempts.groupby(
    ['draft_year', 'nfl_id', 'drill_type', 'drill_name'],
    as_index=False,
    sort=False,
    observed=True,
).agg(
    combine_peak_speed=('attempt_peak_speed', 'median'),
    combine_mean_acceleration=('attempt_mean_acceleration', 'mean'),
    attempt_count=('attempt', 'nunique'),
)
del sensor, attempts

game_year = games.set_index('game_id')['season'].to_dict()
game_type = games.set_index('game_id')['season_type'].to_dict()
rookie_stats = {}
for year in sorted(combine['draft_year'].unique()):
    tracking_path = DATA_DIR / f'game_tracking_{int(year)}.csv'
    if not tracking_path.is_file():
        raise FileNotFoundError(f'Missing rookie-year tracking file: {tracking_path.name}')
    rookie_game_ids = {
        int(game_id)
        for game_id, season in game_year.items()
        if season == year and game_type.get(game_id) == 'REG'
    }
    cohort_ids = set(combine.loc[combine.draft_year.eq(year), 'nfl_id'].astype(int))
    yearly_stats = {}
    for chunk in pd.read_csv(
        tracking_path,
        usecols=['game_id', 'nfl_id', 's', 'a'],
        chunksize=200_000,
    ):
        mask = chunk['game_id'].isin(rookie_game_ids) & chunk['nfl_id'].isin(cohort_ids)
        selected = chunk.loc[mask]
        if selected.empty:
            continue
        grouped = selected.groupby('nfl_id', sort=False).agg(
            frame_count=('nfl_id', 'size'),
            speed_count=('s', 'count'),
            speed_sum=('s', 'sum'),
            mean_acceleration_count=('a', 'count'),
            acceleration_sum=('a', 'sum'),
        )
        for nfl_id, values in grouped.iterrows():
            key = int(nfl_id)
            state = yearly_stats.setdefault(
                key,
                {'frame_count': 0, 'speed_count': 0, 'speed_sum': 0.0,
                 'acceleration_count': 0, 'acceleration_sum': 0.0},
            )
            state['frame_count'] += int(values.frame_count)
            state['speed_count'] += int(values.speed_count)
            state['speed_sum'] += float(values.speed_sum)
            state['acceleration_count'] += int(values.mean_acceleration_count)
            state['acceleration_sum'] += float(values.acceleration_sum)
    rookie_stats.update({(year, nfl_id): value for nfl_id, value in yearly_stats.items()})
    print(f'{year}: rookie REG tracking summarized for {len(yearly_stats):,} Combine players')

rookie = pd.DataFrame([
    {
        'draft_year': year,
        'nfl_id': nfl_id,
        'rookie_regular_season_frames': value['frame_count'],
        'rookie_mean_speed': value['speed_sum'] / value['speed_count'] if value['speed_count'] else np.nan,
        'rookie_mean_acceleration': value['acceleration_sum'] / value['acceleration_count'] if value['acceleration_count'] else np.nan,
    }
    for (year, nfl_id), value in rookie_stats.items()
])
career['career_total_snaps'] = career[
    ['career_offensive_snaps', 'career_defensive_snaps', 'career_special_teams_snaps']
].sum(axis=1, min_count=1)
candidate = combine_features.merge(
    combine[['draft_year', 'nfl_id', 'combine_position', 'forty']],
    on=['draft_year', 'nfl_id'], how='inner', validate='many_to_one',
).merge(rookie, on=['draft_year', 'nfl_id'], how='left', validate='many_to_one').merge(
    career[['nfl_id', 'career_total_snaps']],
    on='nfl_id', how='left', validate='many_to_one',
)
candidate = candidate.loc[candidate.drill_type.fillna('').str.startswith('SKILL_DRILLS_')].copy()
candidate['complete_case'] = (
    candidate.career_total_snaps.notna()
    & candidate.rookie_mean_speed.notna()
    & candidate.rookie_regular_season_frames.ge(100)
)
coverage = candidate.groupby(
    ['combine_position', 'drill_type', 'drill_name', 'draft_year'],
    as_index=False, observed=True,
).agg(complete_players=('complete_case', 'sum'))
ranked = coverage.groupby(
    ['combine_position', 'drill_type', 'drill_name'],
    as_index=False, observed=True,
).agg(complete_players=('complete_players', 'sum')).sort_values(
    ['complete_players', 'combine_position', 'drill_type', 'drill_name'],
    ascending=[False, True, True, True],
)
winner = ranked.iloc[0]
POSITION = str(winner.combine_position)
DRILL_TYPE = str(winner.drill_type)
DRILL_NAME = str(winner.drill_name)
selected = candidate.loc[
    candidate.combine_position.eq(POSITION)
    & candidate.drill_type.eq(DRILL_TYPE)
    & candidate.drill_name.eq(DRILL_NAME)
    & candidate.complete_case
].sort_values(['draft_year', 'nfl_id']).reset_index(drop=True)
assert not selected.duplicated(['draft_year', 'nfl_id']).any()
assert (POSITION, DRILL_TYPE, DRILL_NAME) == ('DL', 'SKILL_DRILLS_DL', 'PASS_RUSH_DRILL')
cohort_counts = selected.groupby('draft_year').nfl_id.nunique().rename('players').reset_index()
display(cohort_counts)
print(f'Selected cohort: {POSITION} / {DRILL_TYPE} / {DRILL_NAME}; n={len(selected)}')

## Cohort-adjusted associations

The association uses rank-transformed values with draft-year indicators partialled out by least squares. Bootstrap intervals resample players within draft-year strata (1,000 replicates). The 40-yard comparison uses the same complete-case players as the partial analysis. These are descriptive associations; no model is trained and no causal or out-of-sample prediction claim is made.

In [ ]:
def rank_residuals(frame, value_column, controls):
    ranks = frame[value_column].rank(method='average', pct=True).to_numpy(dtype=float)
    design_parts = [np.ones((len(frame), 1), dtype=float)]
    cohort_dummies = pd.get_dummies(frame.draft_year.astype(str), drop_first=True, dtype=float)
    if not cohort_dummies.empty:
        design_parts.append(cohort_dummies.to_numpy(dtype=float))
    for column in controls:
        design_parts.append(frame[column].rank(method='average', pct=True).to_numpy(dtype=float)[:, None])
    design = np.column_stack(design_parts)
    coefficients, *_ = np.linalg.lstsq(design, ranks, rcond=None)
    return ranks - design @ coefficients

def adjusted_rank_correlation(frame, x_column, y_column, controls=None):
    controls = controls or []
    data = frame[[x_column, y_column, 'draft_year', *controls]].dropna()
    if len(data) < 3:
        return float('nan')
    x = rank_residuals(data, x_column, controls)
    y = rank_residuals(data, y_column, controls)
    if np.std(x) == 0 or np.std(y) == 0:
        return float('nan')
    return float(np.corrcoef(x, y)[0, 1])

def bootstrap_interval(frame, x_column, y_column, controls=None, replicates=1000, seed=20271011):
    controls = controls or []
    data = frame.dropna(subset=[x_column, y_column, 'draft_year', *controls])
    rng = np.random.default_rng(seed)
    groups = [group.reset_index(drop=True) for _, group in data.groupby('draft_year', sort=True)]
    estimates = []
    for _ in range(replicates):
        sample = pd.concat(
            [group.iloc[rng.integers(0, len(group), size=len(group))] for group in groups],
            ignore_index=True,
        )
        value = adjusted_rank_correlation(sample, x_column, y_column, controls)
        if np.isfinite(value):
            estimates.append(value)
    return tuple(float(value) for value in np.quantile(estimates, [0.025, 0.975]))

relationships = [
    ('Combine peak speed', 'combine_peak_speed', 'Rookie mean speed', 'rookie_mean_speed'),
    ('Combine mean acceleration', 'combine_mean_acceleration', 'Rookie mean acceleration', 'rookie_mean_acceleration'),
    ('Combine peak speed', 'combine_peak_speed', 'Career total snaps', 'career_total_snaps'),
    ('Forty-yard time', 'forty', 'Rookie mean speed', 'rookie_mean_speed'),
]
results = []
for index, (feature_label, feature, outcome_label, outcome) in enumerate(relationships):
    valid = selected.dropna(subset=[feature, outcome, 'draft_year'])
    low, high = bootstrap_interval(valid, feature, outcome, seed=20271011 + index)
    result = {
        'feature': feature_label,
        'outcome': outcome_label,
        'players': len(valid),
        'cohort_adjusted_rank_correlation': adjusted_rank_correlation(valid, feature, outcome),
        'bootstrap_95pct_low': low,
        'bootstrap_95pct_high': high,
    }
    if feature == 'combine_peak_speed' and outcome == 'rookie_mean_speed':
        comparable = valid.dropna(subset=['forty'])
        result['partial_controlling_for_forty'] = adjusted_rank_correlation(
            comparable, feature, outcome, controls=['forty']
        )
        partial_low, partial_high = bootstrap_interval(
            comparable, feature, outcome, controls=['forty'],
            seed=20272011,
        )
        result['partial_95pct_low'] = partial_low
        result['partial_95pct_high'] = partial_high
        result['partial_players'] = len(comparable)
    results.append(result)
associations = pd.DataFrame(results)
display(associations.round(3))
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path(os.environ.get('GRIDIRON_IQ_OUTPUT_DIR', Path.cwd() / 'gridiron_iq_outputs'))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
selected.to_csv(OUTPUT_DIR / 'selected_player_analysis.csv', index=False)
associations.to_csv(OUTPUT_DIR / 'associations.csv', index=False)

colors = {2023: '#2864a5', 2024: '#d17b0f', 2025: '#35835a'}
fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
for year, group in selected.groupby('draft_year', sort=True):
    ax.scatter(
        group.combine_peak_speed, group.rookie_mean_speed,
        label=str(int(year)), color=colors.get(int(year), '#555555'),
        alpha=0.78, edgecolors='white', linewidths=0.4, s=42,
    )
ax.set_xlabel('Median attempt-level Combine peak speed (source units)')
ax.set_ylabel('Rookie regular-season mean tracked speed (source units)')
ax.set_title('DL | Pass Rush Drill: Combine to rookie tracking')
ax.grid(alpha=0.2)
ax.legend(title='Draft cohort', frameon=False)
fig.savefig(OUTPUT_DIR / 'combine_to_rookie_tracking.png', dpi=180)
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 4.5), constrained_layout=True)
ax.bar(cohort_counts.draft_year.astype(str), cohort_counts.players, color='#2864a5')
ax.set_xlabel('Draft cohort')
ax.set_ylabel('Complete-case players')
ax.set_title('DL | Pass Rush Drill: cohort coverage')
ax.grid(axis='y', alpha=0.2)
fig.savefig(OUTPUT_DIR / 'cohort_coverage.png', dpi=180)
plt.show()
plt.close(fig)
print(f'Figures and analysis CSVs saved to {OUTPUT_DIR}')

## Interpretation and safeguards

The primary sensor feature is explicitly linked to a subsequent, draft-year regular-season tracking outcome through `nfl_id` and the `games.csv` season/game mapping. The measured relationship is associative only. Rookie mean speed depends on assignments, plays, playing time, and tracking availability; cohort adjustment does not remove those influences. Career snaps have unequal follow-up and are secondary only. The selected drill is chosen by complete-case coverage, not by correlation size. The figures and CSVs are saved under `OUTPUT_DIR` for download or inclusion in the Kaggle Writeup.